In [17]:
from datetime import datetime, timedelta
import locale
import os
import pytz
import pandas as pd
from jinja2 import Environment, FileSystemLoader
import weasyprint

locale.setlocale(locale.LC_ALL, 'pt_br')

data_hoje = datetime.now(pytz.timezone('Brazil/East'))

# print(data_hoje.strftime('%Y-%m-%dT%H:%M:%S%:z'))
# print(data_hoje.strftime('%B').title())

data_m_passado = data_hoje - timedelta(days=30)
# print(data_m_passado.strftime('%Y-%m'))

data_m_futuro = data_hoje + timedelta(days=30)
# print(data_m_futuro.strftime('%B').title())

# Checa se a pasta existe, se não, cria
path = r'C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\Internacional' + '\\' + data_hoje.strftime('%y-%m-%b').title()
if not os.path.exists(path):
  os.mkdir(path)
  os.mkdir(path+r'\NF')
  print("Pasta %s criada!" % path)
else:
  print("Pasta %s já existe" % path)

def find_header_row(file_path):
    # Lê o arquivo Excel inteiro sem cabeçalho
    df = pd.read_excel(file_path, header=None)
    
    # Verifica linha por linha para identificar o cabeçalho
    expected_columns = ['Data da transação', 'Lançamentos', 'Valor', 'Moeda', 'Valor (US$)', 'Cotação/Dia (R$)', 'Valor em R$', 'Despesa no Exterior']
    
    for i, row in df.iterrows():
        if all(col in row.values for col in expected_columns):
            return i
    return None

# Lista de meses (atual e futuro) para iterar
meses = [data_hoje.strftime('%B').title(), data_m_futuro.strftime('%B').title()]

# Dicionário para armazenar os DataFrames dinamicamente
dfs = {}

# Caminho para o arquivo Excel para cada mês
for i, mes in enumerate(meses):
    file_path = r'C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\Internacional' + '\\' + data_hoje.strftime('%y-%m-%b').title() + rf"\Fatura_{mes}.xlsx"
    try:
        # Procura a linha do cabeçalho
        header_row = find_header_row(file_path)
    
        if header_row is not None:
            # Recarrega o arquivo usando a linha correta como cabeçalho
            df = pd.read_excel(file_path, header=header_row)
            dfs[i] = df  # Armazena o DataFrame no dicionário
            # print(f"Cabeçalho encontrado na linha {header_row + 1} para o mês {mes}.")
            # print(dfs[i].head())  # Mostra as primeiras linhas do DataFrame
        else:
            print(f"Cabeçalho não encontrado para o mês {mes}.")
    except FileNotFoundError:
        print(f"Arquivo para o mês {mes} não encontrado.")
        continue    

def clean_data(df, data_m_passado):
    # Remove a coluna 'Unnamed: 0', se ela existir
    df = df.drop(columns=['Unnamed: 0'], errors='ignore')  # Usa 'ignore' para evitar erro se a coluna não existir
    # Remove linhas com dados ausentes na coluna 'Moeda'
    df = df.dropna(subset=['Moeda'])
    # Garante que a coluna 'Data da transação' esteja no formato datetime
    df['Data da transação'] = pd.to_datetime(df['Data da transação'], errors='coerce')
    # Converte a coluna 'Valor' para o tipo float32
    df = df.astype({'Valor': 'float32'})
    # Filtra as linhas onde 'Despesa no Exterior' é "Sim"
    df = df[df['Despesa no Exterior'] == "Sim"]
    # Filtra as linhas com base no mês da 'Data da transação', comparando com o mês passado
    df = df[df['Data da transação'].dt.month == data_hoje.month]
    # Remove as colunas 'Valor (US$)', 'Cotação/Dia (R$)', 'Valor em R$', 'Despesa no Exterior', 'Moeda'
    # df = df.drop(columns=['Valor (US$)', 'Cotação/Dia (R$)', 'Valor em R$', 'Despesa no Exterior', 'Moeda'], errors='ignore')
    df = df.drop(columns=['Despesa no Exterior'], errors='ignore')
    
    # Função para categorizar os lançamentos com base em palavras-chave e estrutura
    def Categoria(Lançamentos):
        """
        Transforma os lançamentos com base nos seguintes exemplos:
            Lançamentos                    Output
        1: "Uber   *UBER   *TRIP"      => "Uber"
        2: "BarELanchonete"            => "Refeição"
        3: "UBER* TRIP"                => "Uber"
        4: "Uber   *UBER   *TRIP"      => "Uber"
        5: "GIRAFFAS BOUGAINVILLE"     => "Refeição"
        6: "TCHANBURGUER"              => "Refeição"
        7: "IFD*OUTBACK STEAKHOUSE RE" => "Refeição"
        8: "GOLDEN LIS HOTEIS LTDA"    => "Refeição"
        9: "MAIS SABOR"                => "Refeição"
        10: "POSTO JATAI"              => "Posto"
        11: "POSTO A4"                 => "Posto"
        """
        # Busca todas as letras maiúsculas no lançamento
        # findall1 = regex.findall(r"\p{Lu}", Lançamentos)
        
        # Verifica se a palavra "Uber" está na string, ignorando maiúsculas e minúsculas
        if "uber" in Lançamentos.lower():
            return "Uber"
        
        # Verifica se a palavra "park" ou "estacionamento" está presente, ignorando maiúsculas/minúsculas
        if "park" in Lançamentos.lower() or "estacionamento" in Lançamentos.lower():
            return "Estacionamento"

        # Verifica se o número de letras maiúsculas é 6, 8 ou 10
        # if len(findall1) == 6 or len(findall1) == 8 or len(findall1) == 10:
        #     # Encontra o primeiro caractere não alfanumérico cercado por espaços
        #     index1 = [i for i in regex.finditer(r"\p{Zs}*\W\p{Zs}*", Lançamentos)][0].start()
        #     return Lançamentos[:index1].title()
        
        if "posto" in Lançamentos.lower():
            return "Posto"
        
        # Caso contrário, classifica como "Refeição"
        return "Refeição"
    
    # Adiciona a nova coluna "Categoria" usando a função 'Categoria'
    df.insert(2, "Categoria", df.apply(lambda row: Categoria(row["Lançamentos"]), axis=1))
    
    return df

# Aplicar a função clean_data a cada DataFrame no dicionário dfs e concatenar os resultados
df_concatenado = pd.concat([clean_data(df.copy(), data_m_passado) for df in dfs.values()], ignore_index=True)

# Salvar o DataFrame concatenado em um arquivo .xlsx para análise/correções
df_concatenado.to_excel(r'C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\Internacional' + '\\' + data_hoje.strftime('%y-%m-%b').title() + r'\df_concatenado_categorias.xlsx', index=False, engine='openpyxl')

# Mensagem para abrir o arquivo .xlsx
print(r'Arquivo df_concatenado_categorias.xlsx criado com sucesso em C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\Internacional' + '\\' + data_hoje.strftime('%y-%m-%b').title() + r'\df_concatenado_categorias.xlsx')

Pasta C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\Internacional\25-03-Mar já existe
Arquivo para o mês Março não encontrado.
Arquivo df_concatenado_categorias.xlsx criado com sucesso em C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\Internacional\25-03-Mar\df_concatenado_categorias.xlsx


Corrija manualmente a classificação da planilha de gastos!!!

In [29]:
df_corrigido = pd.read_excel(r'C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\Internacional' + '\\' + data_hoje.strftime('%y-%m-%b').title() +r"\df_concatenado_categorias.xlsx")
def clean_data(df_corrigido):
 # Converte as colunas para tipo float (evitando perda de precisão com float32)
    colunas_real = ['Cotação/Dia (R$)', 'Valor em R$']
    colunas_dolar = ['Valor', 'Valor (US$)']
    colunas_numericas = colunas_dolar + colunas_real
    df_corrigido[colunas_numericas] = df_corrigido[colunas_numericas].astype('float64')

    # Arredonda as colunas para duas casas decimais
    df_corrigido[colunas_numericas] = df_corrigido[colunas_numericas].round(2)
    
    # Round column 'Valor' (Number of decimals: 2)
    df_corrigido_n = df_corrigido.copy()  # Cópia para manter os valores numéricos
    df_corrigido_n = df_corrigido_n.round({'Valor': 2})

    locale.setlocale(locale.LC_ALL, 'en_US')

    # Aplica formatação monetária
    for coluna in colunas_dolar:
        df_corrigido[coluna] = df_corrigido[coluna].apply(lambda x: locale.currency(x, grouping=True))

    locale.setlocale(locale.LC_ALL, 'pt_br')

    # Aplica formatação monetária
    for coluna in colunas_real:
        df_corrigido[coluna] = df_corrigido[coluna].apply(lambda x: locale.currency(x, grouping=True))

    # Certificar-se de que 'Data da transação' está no formato de data
    df_corrigido['Data da transação'] = pd.to_datetime(df_corrigido['Data da transação'])
    df_corrigido['Data da transação'] = df_corrigido['Data da transação'].dt.strftime('%d/%m/%Y')

    return df_corrigido, df_corrigido_n

df_corrigido_limpo = clean_data(df_corrigido.copy())

locale.setlocale(locale.LC_ALL, 'en_US')
total = locale.currency(df_corrigido_limpo[1]['Valor'].sum(), grouping=True)
locale.setlocale(locale.LC_ALL, 'pt_br')

data_inicial = df_corrigido_limpo[1]['Data da transação'].min()
data_final = df_corrigido_limpo[1]['Data da transação'].max()

imagens = (
    df_corrigido_limpo[1][df_corrigido_limpo[1]['Categoria'] != "Uber"]
    .apply(lambda row: f"{row['Lançamentos']} {locale.currency(row['Valor'], grouping=True)} {row['Data da transação'].strftime('%d/%m/%Y')}", axis=1)
    .to_list()
)
def summarize_data(df_clean):
    # Agrupar por categoria e calcular somas e quantidades para todas as colunas relevantes
    df_summary = df_clean.groupby(['Categoria']).agg(
        Soma_Valor=('Valor', 'sum'),
        Soma_Valor_USD=('Valor (US$)', 'sum'),
        Soma_Valor_BRL=('Valor em R$', 'sum'),
        Quantidade=('Lançamentos', 'count')
    ).reset_index()
    
    # Calcular médias corretamente para cada moeda
    df_summary['Média_USD'] = df_summary['Soma_Valor_USD'] / df_summary['Quantidade']
    df_summary['Média_BRL'] = df_summary['Soma_Valor_BRL'] / df_summary['Quantidade']
    
    # Arredondamento
    df_summary = df_summary.round({'Soma_Valor_USD': 2, 'Soma_Valor_BRL': 2, 'Média_USD': 2, 'Média_BRL': 2})
    
    # Formatação correta das colunas monetárias
    locale.setlocale(locale.LC_ALL, 'en_US')
    df_summary['Soma_Valor'] = df_summary['Soma_Valor'].apply(lambda x: locale.currency(x, grouping=True, international=True))
    df_summary['Soma_Valor_USD'] = df_summary['Soma_Valor_USD'].apply(lambda x: locale.currency(x, grouping=True))
    df_summary['Média_USD'] = df_summary['Média_USD'].apply(lambda x: locale.currency(x, grouping=True))
    
    locale.setlocale(locale.LC_ALL, 'pt_BR')
    df_summary['Soma_Valor_BRL'] = df_summary['Soma_Valor_BRL'].apply(lambda x: locale.currency(x, grouping=True))
    df_summary['Média_BRL'] = df_summary['Média_BRL'].apply(lambda x: locale.currency(x, grouping=True))
    
    # Renomear a colunas 
    df_summary = df_summary.rename(columns={'Soma_Valor': 'Soma Valores',
                                            'Soma_Valor_USD': 'Soma Valores (US$)',
                                            'Média_USD': 'Média Valores (US$)',
                                            'Soma_Valor_BRL': 'Soma Valores (R$)',
                                            'Média_BRL': 'Média Valores (R$)'})

    return df_summary


df_resumido = summarize_data(df_corrigido_limpo[1].copy())
df_resumido.head()
# Função para carregar imagens da pasta
def load_images_from_folder(folder_path):
    image_paths = [os.path.abspath(os.path.join(folder_path, f)) for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
    return image_paths


# Função para renderizar HTML com dados e imagens
def render_html(data, data_a, data_inicial, data_final, columns, columns_a, images, images_names, datas, template_file, total):
    env = Environment(loader=FileSystemLoader('.'))
    env.globals['zip'] = zip  # Adiciona a função zip ao ambiente Jinja2
    template = env.get_template(template_file)
    html_content = template.render(data=data, data_a=data_a, columns=columns, columns_a=columns_a, images=images, images_names=images_names, datas=datas, data_inicial=data_inicial, data_final=data_final, total=total)
    return html_content

# Função para salvar o HTML como PDF
def save_html_as_pdf(html_content, output_file):
    weasyprint.HTML(string=html_content).write_pdf(output_file)

# Caminho da pasta de imagens
image_folder_path = r'C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\Internacional' + '\\' + data_hoje.strftime('%y-%m-%b').title() + r"\NF"

# Carregar dados do Excel
df = df_corrigido_limpo[0]

# Carregar imagens da pasta
image_paths = load_images_from_folder(image_folder_path)

# Renderizar HTML
html_content = render_html(df.values.tolist(), df_resumido.values.tolist(), data_inicial.strftime('%d/' + '%m/' + '%Y'), data_final.strftime('%d/' + '%m/' + '%Y'), df.columns.tolist(), df_resumido.columns.tolist(), image_paths, imagens, data_hoje.strftime('%B %Y').title(), 'template.html', total)

# Salvar HTML como arquivo PDF
output_pdf_path = 'relatorio.pdf'
save_html_as_pdf(html_content, output_pdf_path)

# Se quiser, também pode salvar como arquivo HTML para visualização antes da conversão
with open(r'C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\Internacional' + '\\' + data_hoje.strftime('%y-%m-%b').title() + r'\relatorio.html', 'w', encoding='utf-8') as f:
    f.write(html_content)


In [13]:
import os
from bs4 import Script
from jinja2 import Environment, FileSystemLoader
from narwhals import String
from weasyprint import HTML

# Função para carregar imagens da pasta no formato correto
def load_images_from_folder(folder_path):
    image_paths = [
        os.path.abspath(os.path.join(folder_path, f)).replace("\\", "/")
        for f in os.listdir(folder_path)
        if f.endswith(('.png', '.jpg', '.jpeg'))
    ]
    return image_paths

# Definição do caminho da pasta de imagens
image_folder_path = os.path.join(
    r'C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas',
    data_hoje.strftime('%y-%m-%b').title(),
    "NF"
).replace("\\", "/")

# Carregar imagens no formato adequado
image_paths = load_images_from_folder(image_folder_path)

def generate_pdf(output_file, template_name='template.html', context=None):
    context = contexto or {}
    # Configura o diretório onde estão os templates
    templates_dir = r'C:\Users\vital\Documents\programas\Relatorio_Financeiro_Mensal'
    env = Environment(loader=FileSystemLoader(templates_dir))

    env.globals['zip'] = zip  # Adiciona a função zip ao ambiente Jinja2

    template = env.get_template(template_name)
    html_content = template.render(context)

    # Cria um arquivo temporário para o HTML renderizado
    temp_html_path = os.path.join(templates_dir, 'temp_rendered.html')
    with open(temp_html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    # Cria o caminho para o PDF de saída
    output_dir = r'C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas' + '\\' + data_hoje.strftime('%y-%m-%b').title()
    os.makedirs(output_dir, exist_ok=True)  # Garante que o diretório de saída exista
    pdf_path = os.path.join(output_dir, output_file)

    HTML(filename=temp_html_path).write_pdf(pdf_path)
    print(f"PDF gerado em: {pdf_path}")

    # (Opcional) Exclua o arquivo temporário depois
    #os.remove(temp_html_path)

# Dados genéricos
# data, data_a, data_inicial, data_final, columns, columns_a, images, images_names, datas, template_file, total
# df.values.tolist(), df_resumido.values.tolist(), data_inicial.strftime('%d/' + '%m/' + '%Y'), data_final.strftime('%d/' + '%m/' + '%Y'), 
# df.columns.tolist(), df_resumido.columns.tolist(), image_paths, imagens, data_m_passado.strftime('%B %Y').title(), 'template.html', total

contexto = {
    "data": df.values.tolist(),
    "data_a": df_resumido.values.tolist(),
    "data_inicial": data_inicial.strftime('%d/' + '%m/' + '%Y'),
    "data_final": data_final.strftime('%d/' + '%m/' + '%Y'),
    "columns": df.columns.tolist(),
    "columns_a": df_resumido.columns.tolist(),
    "images": image_paths,
    "images_names": imagens,
    "datas": data_hoje.strftime('%B %Y').title(),
    "total": total,
}

generate_pdf("relatorio_generico_t4.pdf")

PDF gerado em: C:\Users\vital\OneDrive\Documentos\Trabalho\Despesas\25-02-Fev\relatorio_generico_t4.pdf


In [10]:
context

{'data': [['04/02/2025', 'G LAPA GIRAFFAS CGH', 'Refeição', 'R$ 44,50'],
  ['05/02/2025', 'VILA VELHA HOTEIS E TU', 'Refeição', 'R$ 43,00'],
  ['05/02/2025', 'WEASTCOAST', 'Refeição', 'R$ 39,00'],
  ['05/02/2025', 'AMALFI COMERCIO DE ALI', 'Refeição', 'R$ 50,40'],
  ['06/02/2025', 'GCC COM DE ALIMENTO', 'Refeição', 'R$ 35,80'],
  ['07/02/2025', 'HOTEL ST EMILION', 'Estacionamento', 'R$ 80,00'],
  ['07/02/2025', 'HOTEL ST EMILION', 'Refeição', 'R$ 64,00'],
  ['07/02/2025', 'TIC POSTO LTDA', 'Posto', 'R$ 207,98'],
  ['11/02/2025', 'UBER * PENDING', 'Uber', 'R$ 58,34']],
 'data_a': [['Estacionamento', 'R$ 80,00', 1, 'R$ 80,00'],
  ['Posto', 'R$ 207,98', 1, 'R$ 207,98'],
  ['Refeição', 'R$ 276,70', 6, 'R$ 46,12'],
  ['Uber', 'R$ 58,34', 1, 'R$ 58,34']],
 'data_inicial': '04/02/2025',
 'data_final': '11/02/2025',
 'columns': ['Data da transação', 'Lançamentos', 'Categoria', 'Valor'],
 'columns_a': ['Categoria', 'Soma Valores', 'Quantidade', 'Média'],
 'images': ['C:/Users/vital/OneDrive/Doc